# Introduction to Asset Returns
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Define and compute total returns** — Build returns from price and dividend data, recognizing that "return" is the normalized gain relative to the initial price
2. **Strip out the risk-free rate** — Convert raw returns to *excess returns* so that every subsequent statistic measures compensation over a truly "safe" alternative
3. **Introduce the idea of a risk premium** — Understand that the expected value of excess return is the reward for bearing risk
4. **Motivate the need for a factor model** — See why differing betas demand a formal model and set the stage for multi-factor extensions

## 📋 Table of Contents

1. [Setup](#setup)
2. [What is a Return?](#what-is-a-return)
3. [Decomposing Returns](#decomposing-returns)
4. [Excess Returns](#excess-returns)
5. [Risk Premiums](#risk-premiums)
6. [Stripping the Common Factor](#stripping-the-common-factor)
7. [Exercises](#exercises)
8. [Key Takeaways](#key-takeaways)

---

## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title 🛠️ Setup: Run this cell first (click to expand)

# Uncomment the following line if running in Colab and you need wrds
# !pip install wrds

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Set consistent plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully!")

In [ ]:
#@title 🔧 Helper Functions for WRDS (click to expand)

# Functions that we will use in this notebook that you don't need to worry about for now

def get_daily_wrds_single_ticker(ticker, conn, dividends=True):
    """Get daily price and dividend data for a single ticker from WRDS."""
    tickers = [ticker]
    # Retrieve PERMNOs for the specified tickers
    permnos = conn.get_table(library='crsp', table='stocknames', columns=['permno', 'ticker', 'namedt', 'nameenddt'])
    permnos['nameenddt'] = pd.to_datetime(permnos['nameenddt'])
    permnos = permnos[(permnos['ticker'].isin(tickers)) & (permnos['nameenddt'] == permnos['nameenddt'].max())]
    # Extract unique PERMNOs
    permno_list = [permnos['permno'].unique().tolist()[0]]
    print(permno_list)

    # Query daily stock file for the specified PERMNOs
    query = f"""
        SELECT permno, date, ret, retx, prc       
        FROM crsp.dsf
        WHERE permno IN ({','.join(map(str, permno_list))})
        ORDER BY date
    """
    daily_returns = conn.raw_sql(query, date_cols=['date'])
    daily_returns = daily_returns.merge(permnos[['permno', 'ticker']], on='permno', how='left')
    
    if dividends:
        daily_returns['D'] = (daily_returns.ret - daily_returns.retx) * daily_returns.prc.abs().shift(1)
        daily_returns['P'] = daily_returns.prc.abs()
        daily_returns = daily_returns[['date', 'P', 'D']].set_index('date').dropna()
    else:
        daily_returns = daily_returns[['date', 'ret']].set_index('date').dropna()    

    return daily_returns


def get_daily_wrds(conn, tickers=None):
    """Get daily return data for multiple tickers from WRDS."""
    # Retrieve PERMNOs for the specified tickers
    permnos = conn.get_table(library='crsp', table='stocknames', columns=['permno', 'ticker', 'namedt', 'nameenddt'])
    permnos['nameenddt'] = pd.to_datetime(permnos['nameenddt'])
    permnos = permnos[(permnos['ticker'].isin(tickers)) & (permnos['nameenddt'] == permnos['nameenddt'].max())]
    # Extract unique PERMNOs
    permno_list = permnos['permno'].unique().tolist()
    print(permno_list)

    # Query daily stock file for the specified PERMNOs
    query = f"""
        SELECT permno, date, ret, retx, prc       
        FROM crsp.dsf
        WHERE permno IN ({','.join(map(str, permno_list))})
        ORDER BY date, permno
    """
    daily_returns = conn.raw_sql(query, date_cols=['date'])
    daily_returns = daily_returns.merge(permnos[['permno', 'ticker']], on='permno', how='left')
    # Pivot data to have dates as index and tickers as columns
    daily_returns = daily_returns.pivot(index='date', columns='ticker', values='ret')    
    daily_returns = daily_returns[tickers]

    return daily_returns

---

## What is a Return? <a id="what-is-a-return"></a>

### The Return Formula

Let's say you paid $P_t$ at date $t$ for an asset. At date $t+1$ the price is $P_{t+1}$ and you earn some dividend $D_{t+1}$.

Then we say that your **return** is:

$$R_{t+1} = \frac{P_{t+1} + D_{t+1} - P_t}{P_t}$$

It is the gain you made (everything you got at date $t+1$), divided by how much you put in (the price of the asset).

> **💡 Key Insight:**
>
> This definition works for **ANY asset** that has a positive price:
> - Stocks, bonds, commodities, crypto, most real assets
> - The return simply normalizes the "dollar gain" by the cost of the asset
>
> In practice there are many types of distributions that are economically like a dividend but have different names: cash dividends, stock dividends, capital gain distributions, rights offerings, acquisition-related distributions, splits.

### Loading Price and Dividend Data

Let's start by loading price and dividend data on a single stock.

In [ ]:
# If you have WRDS access, you can connect to the database:
# import wrds
# conn = wrds.Connection()
# df = get_daily_wrds_single_ticker('UNH', conn)

> **📌 Remember:**
>
> Since most of you probably don't have the WRDS setup yet, I have saved the data for UNH for you to access directly.

In [ ]:
# Load UNH data directly from GitHub
df = pd.read_csv('https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/UNH_data.csv')
df.date = pd.to_datetime(df.date)
df.set_index('date', inplace=True)

print(f"Data range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Number of observations: {len(df)}")
df.head()

### Computing Returns

> **🔧 Exercise:**
>
> How do you construct returns from price and dividend data?

In [ ]:
# Compute returns using the formula: R = (P_t + D_t - P_{t-1}) / P_{t-1}
df['ret'] = (df['P'] + df['D'] - df['P'].shift(1)) / df['P'].shift(1)

df.head(10)

In [ ]:
# Average daily return
df['ret'].mean()

If I had invested 100 dollars on a random day in the sample, on average I would have 100.09 dollars the next day.

A return of:
$$\frac{100.09 - 100}{100} = 0.09\%$$

> **🔧 Exercise:**
>
> Suppose that at the start of the sample we invested 1 dollar in this stock and got all the dividends and used them to buy more of the stock:
>
> 1. How many dollars would we have at the end of the sample?
> 2. What is the cumulative return on our investment?
> 3. What is the annualized return?
> 4. What is the dividend yield of the asset?

In [ ]:
# Your code here


> **🤔 Think and Code:**
>
> Suppose we are now at the end of the sample:
>
> 1. You have 1 million dollars invested in this stock. The next day there's a 16% chance your portfolio's value will fall below a certain amount. How would you estimate that amount?
>    - Now suppose you want to know this value in one year. What is your estimate?
>    - What is the EXPECTED value of your holdings in one year? How should you think about estimating this?
>
> 2. What can you plot:
>    - To have a sense of the distribution of 1-day returns? And 1-year returns?
>    - To give you a sense of how these returns vary over time?
>
> 3. What drives these returns? Why do they vary over time?

In [ ]:
# Your code here


---

## Decomposing Returns <a id="decomposing-returns"></a>

Returns of an individual stock can be driven by many things:

- **Time-value of money**: There are periods where you can get very high returns even in perfectly safe assets
- **Common factors**: All stocks went up because of a stronger economy, all stocks went down because of a financial crisis
- **Individual factors**: A new drug/technology that the firm sells. Anything specific to the firm

> **💡 Key Insight:**
>
> When investing, it is essential to understand **where your performance comes from**.
>
> We will build towards a framework of investing that thinks differently about investing in systematic risk vs. idiosyncratic risk.

The first step will be to build this decomposition, which will start now and culminate in our factor models lecture.

---

## Excess Returns <a id="excess-returns"></a>

### Stripping out the Time-Value of Money

In this class we will do a lot of decomposing, but let's start by stripping down the "time-value of money" piece.

We first define an **excess return**: the return minus the risk-free rate

$$\text{excess return}^{\text{stock } i} = \text{return}^{\text{stock } i} - \text{risk-free rate}$$

- We typically use the returns of a 3-month treasury bill to measure the risk-free rate
- This obviously should be currency dependent
- We will denote $rf$ for the risk-free rate
- We often add superscript "e" to denote an excess return
- So if $r^i$ stands for the return of stock $i$, then $r^{e,i}$ is its excess return

> **📌 Remember:**
>
> Conceptually you want to use the risk-free asset for the relevant holding period you are evaluating, but for this class you can think of the "Fed Funds Rate" or the "3-month treasury-bill rate".

In [ ]:
# Load risk-free rate data from FRED
from pandas_datareader.data import DataReader
import datetime

# Define the date range
start_date = datetime.datetime(1960, 1, 1)
end_date = datetime.datetime.now()

# Get 3-month treasury bill rate
df_rf = DataReader("DGS3MO", "fred", start_date, end_date)
df_rf.reset_index(inplace=True)
df_rf.columns = ["Date", "rf"]
df_rf.rf = df_rf.rf / 100  # Convert from percentage to decimal
df_rf.set_index("Date", inplace=True)

df_rf.plot(title='3-Month Treasury Bill Rate')
plt.ylabel('Rate')
plt.show()

> **🤔 Think and Code:**
>
> How do we interpret this rate?
>
> If I invested in the T-bill in 2005, how much money would I have at the end of 3 months? And at the end of the year?

### Merging with Stock Returns

Let's get the return column of our dataframe and merge it together with our risk-free return.

In [ ]:
# Merge stock returns with risk-free rate
df = df.merge(df_rf, left_index=True, right_index=True, how='left')
df.head()

### Computing Excess Returns

Now let's construct another column called 'ret_e' for excess returns.

In [ ]:
# Compute excess returns (need to convert annual rf to daily)
df['ret_e'] = df['ret'] - df['rf'] / 252

df[['ret', 'rf', 'ret_e']].head(10)

> **💡 Concept Check:**
>
> - What is the trading interpretation of such series?
> - Is it the "return" to which strategy exactly?
> - How do the historical distributions of ret and ret_e compare?
> - Are their averages similar?
> - Are their historical standard deviations similar?
> - Is the standard deviation of risk-free rate useful to tell you the distribution of your returns at the end of 3 months for an investment in the risk-free asset?

In [ ]:
# Compare statistics
print("Comparison of Returns vs Excess Returns")
print("━" * 50)
print(f"Mean return:         {df['ret'].mean():>12.6f}")
print(f"Mean excess return:  {df['ret_e'].mean():>12.6f}")
print(f"Mean rf (daily):     {(df['rf']/252).mean():>12.6f}")
print("━" * 50)
print(f"Std return:          {df['ret'].std():>12.6f}")
print(f"Std excess return:   {df['ret_e'].std():>12.6f}")

---

## Risk Premiums <a id="risk-premiums"></a>

We call the **risk premium** what we earn in excess of what a risk-free investment pays.

So the risk premium of an asset is the expected value of the excess return:

$$\text{risk premium of stock } i = E[r^{e,i}]$$

So the expected return of a stock is:

$$E[r^i] = rf + E[r^{e,i}]$$

The realized return is obviously volatile, hence the risk:

$$u_i = r^{e,i} - E[r^{e,i}]$$

will be negative 50% of the time and sometimes quite negative!

> **💡 Key Insight:**
>
> Big picture: the goal of quant investing is to harvest these risk-premia while managing the risk.
>
> - We know the risk-free rate (it is the yield on a short-term government bond)
> - How do we figure out the risk-premium of an asset?

---

## Stripping the Common Factor <a id="stripping-the-common-factor"></a>

### Building a Simple Model

We will build a simple model to help us both think about this risk-premium and also the risk that we need to manage.

It will be useful to decompose the excess returns further to better understand its risk and its premium.

We know that the overall market portfolio moves around, so it is natural to strip that market-wide movement from the stock returns.

Suppose $f$ is this common factor. One possibility is to write:

$$r^i = r^i - rf + rf$$

$$r^i = r^i - rf + rf - f + f$$

Reorganizing, we have that the return can be written as:

$$r^i = rf + f + (r^{e,i} - f)$$

$$r^i = \underbrace{rf}_{\text{risk-free rate}} + \underbrace{f}_{\text{factor}} + \underbrace{(r^{e,i} - f)}_{\text{firm-specific component}}$$

> **🤔 Think and Code:**
>
> - What is this common factor?
> - Will this work? When will it work?

### Using SPY as a Market Proxy

I will use the returns on the SPY ETF as a market proxy:

$$f = \text{return}^{SPY}$$

This fund holds a market-capitalization weighted portfolio of the largest 500 US stocks (roughly) — this consists of about 85% of the total universe of investable US equities.

> **⚠️ Caution:**
>
> The press often cites the **DOW JONES** as another proxy for the overall movement in stocks — but it is a terrible proxy, since it is an equal-weighted portfolio of 30 arbitrarily chosen stocks. **Please never ever use that!**

If you really want to use a portfolio that tracks the entirety of the US stock market universe, you can use **VTI**, which is an ETF that holds a market-capitalization weighted portfolio of all publicly traded US stocks (about $60 trillion investment universe).

For reasons that will be clear later — but don't matter for now — I will also strip the risk-free rate from our common factor:

$$f = \text{return}^{SPY} - rf$$

In [ ]:
# If you have WRDS access:
# conn = wrds.Connection()
# df = get_daily_wrds(conn, tickers=['SPY', 'WMT', 'JPM'])

> **📌 WRDS Workaround:**
>
> You can load this data directly if you don't have WRDS access.

In [ ]:
# Load multi-stock data directly from GitHub
df = pd.read_csv('https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/df_SPYWMTJPM_data.csv')
df.date = pd.to_datetime(df.date)
df.set_index('date', inplace=True)

df.head()

> **🤔 Calculation check:**
>
> Why divide the risk-free rate by 252?

In [ ]:
# Create excess returns
# The rf is annual, so we divide by 252 to get daily
df = df.merge(df_rf / 252, left_index=True, right_index=True, how='left')
df = df.dropna()

df.head()

In [ ]:
# Compute excess returns for all stocks
df_re = df[['SPY', 'WMT', 'JPM']].sub(df['rf'], axis=0)

df_re.head()

### Visualizing Co-movement

> **🔧 Exercise:**
>
> How can we visualize co-movement between stocks?

In [ ]:
# Scatter plots to visualize co-movement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_re.plot.scatter(x='SPY', y='WMT', ax=axes[0], alpha=0.3)
axes[0].set_title('SPY vs WMT Excess Returns')

df_re.plot.scatter(x='SPY', y='JPM', ax=axes[1], alpha=0.3)
axes[1].set_title('SPY vs JPM Excess Returns')

plt.tight_layout()
plt.show()

> **🤔 Think and Code:**
>
> What is a quantitative way to measure co-movement?

In [ ]:
# Correlation matrix
df_re.corr()

### Cleaning Up Co-movement

Let's try to clean the stock from exposure to the common factor by subtracting it out.

I am using 'm_spy' to denote the returns minus the return on the factor.

In [ ]:
# Subtract SPY returns to "clean" the factor exposure
df_re['WMT_m_spy'] = df_re['WMT'] - df_re['SPY']
df_re['JPM_m_spy'] = df_re['JPM'] - df_re['SPY']

df_re[['WMT', 'WMT_m_spy', 'JPM', 'JPM_m_spy']].head()

> **💡 Concept Check:**
>
> - What is the trading interpretation in this case?
> - What is the portfolio that yields the 'ret_m_spy' payoff?
> - What is the theoretical "cost" of implementing this portfolio (if you want to have $100 exposure to it)?
> - In practice, how much capital do you need? What does it depend on?

In [ ]:
# Plot the "cleaned" returns against SPY
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_re.plot.scatter(x='SPY', y='WMT_m_spy', ax=axes[0], alpha=0.3)
axes[0].set_title('SPY vs WMT (minus SPY)')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)

df_re.plot.scatter(x='SPY', y='JPM_m_spy', ax=axes[1], alpha=0.3)
axes[1].set_title('SPY vs JPM (minus SPY)')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Did It Work?

What is going on?

The "model" motivating our decomposition was:

$$r = r - rf - f + f + rf$$

with the $r - f - rf$ picking up the stock-specific movement and $f$ the common movement.

But it seems that the residual still has factor exposure!

- For **Walmart**, the residual is negatively correlated, so it seems we are taking out **too much**
- For **JPMorgan**, it is still positive, so we are taking out **too little**
- Taking out 1 dollar of factor exposure for each 1 dollar of stock exposure reduced the factor exposure and the asset volatilities, but we can do better

> **🤔 Think and Code:**
>
> What should we do? How do we fix this?
>
> *Hint: This motivates the need for a **factor model** with betas!*

---

## 📝 Exercises <a id="exercises"></a>

### Exercise 1: Computing Cumulative Returns

> **🔧 Exercise:**
>
> Using the UNH data:
> 1. Compute the cumulative return from start to end of sample
> 2. Plot the cumulative wealth growth of a $1 investment
> 3. What is the annualized return over the entire period?

In [ ]:
# Your code here


<details>
<summary>💡 Click to see solution</summary>

```python
# Reload UNH data
df_unh = pd.read_csv('https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/UNH_data.csv')
df_unh.date = pd.to_datetime(df_unh.date)
df_unh.set_index('date', inplace=True)
df_unh['ret'] = (df_unh['P'] + df_unh['D'] - df_unh['P'].shift(1)) / df_unh['P'].shift(1)

# Cumulative return
df_unh['cum_ret'] = (1 + df_unh['ret']).cumprod()
total_return = df_unh['cum_ret'].iloc[-1] - 1
print(f"Total cumulative return: {total_return:.2%}")

# Annualized return
n_years = (df_unh.index[-1] - df_unh.index[0]).days / 365.25
annual_return = (1 + total_return) ** (1/n_years) - 1
print(f"Annualized return: {annual_return:.2%}")

# Plot
df_unh['cum_ret'].plot(figsize=(12, 5), title='UNH Cumulative Wealth Growth ($1 Initial)')
plt.ylabel('Wealth')
plt.show()
```
</details>

### Exercise 2: Value at Risk

> **🤔 Think and Code:**
>
> You have $1 million invested in UNH:
> 1. Estimate the 5th percentile of daily returns (Value at Risk proxy)
> 2. What dollar amount could you lose on a "bad day" (5% probability)?
> 3. Find the worst single-day loss in the sample
> 4. Plot the distribution with the 5th percentile marked

In [ ]:
# Your code here


<details>
<summary>💡 Click to see solution</summary>

```python
portfolio_value = 1_000_000

# 5th percentile
var_5 = df_unh['ret'].quantile(0.05)
print(f"5th percentile return: {var_5:.4%}")

# Dollar loss
dollar_loss = portfolio_value * abs(var_5)
print(f"5% daily VaR: ${dollar_loss:,.0f}")

# Worst day
worst_day = df_unh['ret'].min()
worst_date = df_unh['ret'].idxmin()
print(f"Worst day: {worst_day:.4%} on {worst_date.date()}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_unh['ret'].dropna(), bins=100, edgecolor='white', alpha=0.7)
ax.axvline(var_5, color='orange', linestyle='--', linewidth=2, label=f'5% VaR: {var_5:.2%}')
ax.axvline(worst_day, color='red', linestyle='--', linewidth=2, label=f'Worst: {worst_day:.2%}')
ax.legend()
ax.set_title('UNH Return Distribution with Risk Measures')
ax.set_xlabel('Daily Return')
plt.show()
```
</details>

### Exercise 3: Comparing Correlations

> **🔧 Exercise:**
>
> Using the SPY, WMT, JPM data:
> 1. Compute the correlation of each stock with SPY (before cleaning)
> 2. Compute the correlation of the "cleaned" returns (stock - SPY) with SPY
> 3. Explain why the correlations changed the way they did

In [ ]:
# Your code here


<details>
<summary>💡 Click to see solution</summary>

```python
# Original correlations
print("Original correlations with SPY:")
print(f"WMT: {df_re['WMT'].corr(df_re['SPY']):.4f}")
print(f"JPM: {df_re['JPM'].corr(df_re['SPY']):.4f}")

# Cleaned correlations
print("\nCleaned correlations with SPY:")
print(f"WMT_m_spy: {df_re['WMT_m_spy'].corr(df_re['SPY']):.4f}")
print(f"JPM_m_spy: {df_re['JPM_m_spy'].corr(df_re['SPY']):.4f}")

# Explanation: 
# - WMT has a beta < 1, so subtracting SPY removes "too much" market exposure
#   The residual is negatively correlated with SPY
# - JPM has a beta > 1, so subtracting SPY removes "too little" market exposure
#   The residual is still positively correlated with SPY
```
</details>

---

## 🧠 Key Takeaways <a id="key-takeaways"></a>

1. **Return = price change + income, normalized.** Whether for stocks, bonds, or crypto, total return compares what you end with to what you put in.

2. **Excess return is the correct performance yardstick.** Subtracting the risk-free rate lets you focus on compensation for taking risk, not for simply waiting.

3. **Risk premiums are expectations, not guarantees.** The historical average of excess returns is noisy and must be interpreted with care.

4. **Stocks do not share the same market sensitivity.** Removing one dollar of market exposure from each asset under- or over-hedges depending on its true beta.

5. **This motivates factor models.** In the next notebook, we'll learn how to properly estimate betas and build a formal factor model framework.

---

**Next Notebook:** Factor Models — where we'll learn how to properly estimate betas and decompose returns into systematic and idiosyncratic components.